In [1]:
print("Spark session is working")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: ff1a0d4c-f046-4c80-9d58-c450692a0748
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session ff1a0d4c-f046-4c80-9d58-c450692a0748 to get into ready status...
Session ff1a0d4c-f046-4c80-9d58-c450692a0748 has been created.
Spark session is working


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [87]:
BUCKET = "ahmed-data-bucket-2026-new"

RAW_PATH = f"s3://{BUCKET}/raw/ecommerce"
PROCESSED_PATH = f"s3://{BUCKET}/processed/ecommerce"


In [4]:
# 2. Read all raw CSV files
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/customers.csv")
)

categories = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/categories.csv")
)

products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/products.csv")
)

departments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/departments.csv")
)

employees = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/employees.csv")
)

suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/suppliers.csv")
)

orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/orders.csv")
)

order_details = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/order_details.csv")
)

payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/payments.csv")
)

product_suppliers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/product_suppliers.csv")
)

shippers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shippers.csv")
)

shipments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/shipments.csv")
)

In [5]:
print("Customers:", customers.count())
print("Categories:", categories.count())
print("Products:", products.count())
print("Departments:", departments.count())
print("Employees:", employees.count())
print("Suppliers:", suppliers.count())
print("Orders:", orders.count())
print("Order Details:", order_details.count())
print("Payments:", payments.count())
print("Product Suppliers:", product_suppliers.count())
print("Shippers:", shippers.count())
print("Shipments:", shipments.count())

Customers: 10000
Categories: 20
Products: 1000
Departments: 10
Employees: 200
Suppliers: 100
Orders: 50000
Order Details: 100000
Payments: 45000
Product Suppliers: 2027
Shippers: 10
Shipments: 40000


In [7]:
customers.printSchema()
orders.printSchema()
order_details.printSchema()
products.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

root
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Status: string (nullable = true)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: integer (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true

In [8]:
customers.show(10, truncate=False)
orders.show(10, truncate=False)

+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City      |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+----------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza      |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai     |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza      |United Kingdom|2020-12-22 00:00:00|
|4         |Patricia |Miller  |patricia.miller4@example.com |London    |Germany       |2020-05-10 00:00:00|
|5         |Robert   |Johnson |robert.johnson5@example.com  |Cairo     |Saudi Arabia  |2022-06-14 00:00:00|
|6         |Jeffery  |Wagner  |jeffery.wagner6@example.com  |Dubai     |United Kingdom|2020-04-18 00:00:00|
|7         |Anthony  |Gonzal

In [9]:
def clean_column_names(df):
    for column in df.columns:
        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        
        df = df.withColumnRenamed(column, new_name)
    
    return df

In [10]:
customers = clean_column_names(customers)
categories = clean_column_names(categories)
products = clean_column_names(products)
departments = clean_column_names(departments)
employees = clean_column_names(employees)
suppliers = clean_column_names(suppliers)
orders = clean_column_names(orders)
order_details = clean_column_names(order_details)
payments = clean_column_names(payments)
product_suppliers = clean_column_names(product_suppliers)
shippers = clean_column_names(shippers)
shipments = clean_column_names(shipments)

In [11]:
customers.printSchema()

root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: timestamp (nullable = true)


In [12]:
departments.printSchema()

root
 |-- departmentid: integer (nullable = true)
 |-- departmentname: string (nullable = true)


In [13]:
customers = customers.dropDuplicates(["customerid"])

orders = orders.dropDuplicates(["orderid"])

products = products.dropDuplicates(["productid"])

order_details = order_details.dropDuplicates(["orderdetailid"])

In [14]:
customers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in customers.columns
]).show()

+----------+---------+--------+-----+----+-------+----------------+
|customerid|firstname|lastname|email|city|country|registrationdate|
+----------+---------+--------+-----+----+-------+----------------+
|         0|        0|       0|    0|   0|      0|               0|
+----------+---------+--------+-----+----+-------+----------------+


In [15]:
customers = (
    customers
    .withColumn("firstname", F.trim("firstname"))
    .withColumn("lastname", F.trim("lastname"))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("city", F.trim("city"))
    .withColumn("country", F.trim("country"))
)

In [16]:
orders = (
    orders
    .withColumn("status", F.upper(F.trim("status")))
)

In [17]:
orders = (
    orders
    .withColumn(
        "orderid",
        F.col("orderid").cast("long")
    )
    .withColumn(
        "customerid",
        F.col("customerid").cast("long")
    )
    .withColumn(
        "orderdate",
        F.to_date("orderdate")
    )
)

In [18]:
customers = customers.withColumn(
    "valid_email",
    F.when(
        F.col("email").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        ),
        True
    ).otherwise(False)
)

In [19]:
orders = orders.withColumn(
    "status",
    F.when(
        F.col("status").isin(
            "PENDING",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED"
        ),
        F.col("status")
    ).otherwise("UNKNOWN")
)

In [20]:
order_details = order_details.withColumn(
    "total_amount",
    F.col("quantity") * F.col("unitprice")
)

In [25]:
order_details = order_details.withColumn(
    "quantity",
    F.when(
        F.col("quantity") < 0,
        0
    ).otherwise(F.col("quantity"))
)

In [26]:
order_details = order_details.withColumn(
    "unitprice",
    F.when(
        F.col("unitprice") < 0,
        0
    ).otherwise(F.col("unitprice"))
)

In [27]:
orders = (
    orders
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
    .withColumn("day", F.dayofmonth("orderdate"))
    .withColumn("quarter", F.quarter("orderdate"))
    .withColumn("day_of_week", F.dayofweek("orderdate"))
)

In [28]:
order_sales = (
    orders.alias("o")
    .join(
        order_details.alias("od"),
        F.col("o.orderid") == F.col("od.orderid"),
        "inner"
    )
)

In [29]:
order_sales = order_sales.select(
    F.col("o.orderid"),
    F.col("o.customerid"),
    F.col("o.orderdate"),
    F.col("o.status"),
    F.col("od.productid"),
    F.col("od.quantity"),
    F.col("od.unitprice"),
    F.col("od.total_amount")
)

In [30]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Define the window
customer_window = (
    Window
    .partitionBy("customerid")
    .orderBy(F.col("orderdate").desc())
)

# Add row number
last_order_customer = (
    orders
    .withColumn(
        "last_order_rank",
        F.row_number().over(customer_window)
    )
)

# Keep only the latest order for each customer
last_order_customer = (
    last_order_customer
    .filter(F.col("last_order_rank") == 1)
)

last_order_customer.show()

+-------+----------+----------+---------+----+-----+---+-------+-----------+---------------+
|orderid|customerid| orderdate|   status|year|month|day|quarter|day_of_week|last_order_rank|
+-------+----------+----------+---------+----+-----+---+-------+-----------+---------------+
|  17011|         3|2026-04-01|  PENDING|2026|    4|  1|      2|          4|              1|
|   2861|         9|2025-11-29|DELIVERED|2025|   11| 29|      4|          7|              1|
|  33028|        11|2026-01-18|  UNKNOWN|2026|    1| 18|      1|          1|              1|
|  39870|        14|2026-03-13|  PENDING|2026|    3| 13|      1|          6|              1|
|  11899|        17|2026-06-07|DELIVERED|2026|    6|  7|      2|          1|              1|
|  11271|        28|2025-12-05|CANCELLED|2025|   12|  5|      4|          6|              1|
|  36170|        29|2026-03-20|CANCELLED|2026|    3| 20|      1|          6|              1|
|  27776|        30|2026-08-20|  SHIPPED|2026|    8| 20|      3|      

In [31]:
order_details = (
    order_details
    .withColumn(
        "quantity",
        F.when(F.col("quantity") < 0, 0)
         .otherwise(F.col("quantity"))
    )
    .withColumn(
        "unitprice",
        F.when(F.col("unitprice") < 0, 0)
         .otherwise(F.col("unitprice"))
    )
    .withColumn(
        "total_amount",
        F.round(
            F.col("quantity") * F.col("unitprice"),
            2
        )
    )
)

In [32]:
products = (
    products
    .withColumn("productname", F.trim("productname"))
    .withColumn("brand", F.trim("brand"))
    .withColumn(
        "price",
        F.when(F.col("price") <= 0, None)
         .otherwise(F.col("price"))
    )
    .withColumn(
        "cost",
        F.when(F.col("cost") < 0, None)
         .otherwise(F.col("cost"))
    )
)

In [33]:
def cast_ids_to_long(df, columns):
    for column in columns:
        if column in df.columns:
            df = df.withColumn(
                column,
                F.col(column).cast("long")
            )
    return df

In [34]:
def cast_ids_to_long(df, columns):
    for column in columns:
        if column in df.columns:
            df = df.withColumn(
                column,
                F.col(column).cast("long")
            )
    return df

In [35]:
customers = customers.withColumn(
    "full_name",
    F.concat_ws(
        " ",
        F.col("firstname"),
        F.col("lastname")
    )
)

In [36]:
order_values = (
    order_details
    .groupBy("orderid")
    .agg(
        F.round(
            F.sum("total_amount"),
            2
        ).alias("order_total_amount"),

        F.sum("quantity").alias("order_total_quantity")
    )
)

In [40]:
orders_payments = (
    orders.alias("o")
    .join(
        payments.alias("p"),
        F.col("o.orderid") == F.col("p.orderid"),
        "left"
    )
)

In [44]:
order_sales = (
    orders
    .join(
        order_details,
        orders.orderid == order_details.orderid,
        "inner"
    )
    .select(
        orders.orderid,
        orders.customerid,
        orders.orderdate,
        orders.status,
        orders.year,
        orders.month,
        orders.day,
        orders.quarter,
        orders.day_of_week,
        order_details.orderdetailid,
        order_details.productid,
        order_details.quantity,
        order_details.unitprice,
        order_details.discount,
        order_details.total_amount
    )
)


In [45]:
orders_payments.show(10, truncate=False)

+-------+----------+----------+---------+---------+-------------+-------------------+-------+
|orderid|customerid|orderdate |status   |paymentid|paymentmethod|paymentdate        |amount |
+-------+----------+----------+---------+---------+-------------+-------------------+-------+
|1      |8870      |2024-05-03|CANCELLED|null     |null         |null               |null   |
|3      |8030      |2026-01-14|DELIVERED|26668    |Debit Card   |2026-05-18 00:00:00|113.23 |
|5      |4623      |2025-05-26|DELIVERED|27992    |Debit Card   |2024-09-22 00:00:00|4081.62|
|5      |4623      |2025-05-26|DELIVERED|20473    |PayPal       |2025-08-20 00:00:00|1264.43|
|10     |9903      |2025-08-15|CANCELLED|null     |null         |null               |null   |
|12     |2908      |2024-12-25|UNKNOWN  |null     |null         |null               |null   |
|16     |2017      |2024-06-01|DELIVERED|28872    |Bank Transfer|2026-03-28 00:00:00|660.37 |
|16     |2017      |2024-06-01|DELIVERED|10986    |PayPal   

In [46]:
#dim_Cutomers
dim_customer = (
    customers
    .withColumn(
        "customer_sk",
        F.row_number().over(
            Window.orderBy("customerid")
        )
    )
    .withColumn(
        "full_name",
        F.concat_ws(
            " ",
            F.col("firstname"),
            F.col("lastname")
        )
    )
    .select(
        "customer_sk",
        "customerid",
        "firstname",
        "lastname",
        "full_name",
        "email",
        "city",
        "country",
        "registrationdate",
        "valid_email"
    )
)

In [48]:
.withColumn(
    "employee_sk",
    F.row_number().over(
        Window.orderBy("employeeid")
    )
)

SyntaxError: invalid syntax (<stdin>, line 1)


In [49]:
dim_employee = (
    employees
    .join(
        departments,
        employees.departmentid == departments.departmentid,
        "inner"
    )
    .withColumn(
        "employee_sk",
        F.row_number().over(
            Window.orderBy("employeeid")
        )
    )
    .select(
        "employee_sk",
        "employeeid",
        "managerid",
        "departmentname",
        "firstname",
        "lastname",
        "salary",
        "hiredate"
    )
)

In [50]:
dim_product = (
    products
    .join(
        categories,
        products.categoryid == categories.categoryid,
        "inner"
    )
    .withColumn(
        "product_sk",
        F.row_number().over(
            Window.orderBy("productid")
        )
    )
    .select(
        "product_sk",
        "productid",
        "categoryname",
        "productname",
        "brand",
        "price",
        "cost",
        "stock"
    )
)

In [51]:
dim_product.show(10, truncate=False)

+----------+---------+---------------+------------------+------+-------+-------+-----+
|product_sk|productid|categoryname   |productname       |brand |price  |cost   |stock|
+----------+---------+---------------+------------------+------+-------+-------+-----+
|1         |1        |Gaming         |Bose Printer 1    |Bose  |261.16 |140.65 |399  |
|2         |2        |Mobile Phones  |Lenovo Mouse 2    |Lenovo|1874.2 |1460.92|155  |
|3         |3        |Gaming         |Sony Headphones 3 |Sony  |1272.54|846.77 |315  |
|4         |4        |Toys           |LG Smart Watch 4  |LG    |1180.67|637.81 |95   |
|5         |5        |Automotive     |Bose Television 5 |Bose  |253.18 |165.51 |375  |
|6         |6        |Home Appliances|Bose Headphones 6 |Bose  |2070.53|1524.96|256  |
|7         |7        |Furniture      |Apple Headphones 7|Apple |2542.44|1485.25|181  |
|8         |8        |Furniture      |Bose Backpack 8   |Bose  |2231.71|1713.33|195  |
|9         |9        |Garden         |Nike 

In [52]:
dim_order = (
    orders
    .withColumn(
        "order_sk",
        F.row_number().over(
            Window.orderBy("orderid")
        )
    )
    .select(
        "order_sk",
        "orderid"
    )
)

In [53]:
dim_order.show(10, truncate=False)

+--------+-------+
|order_sk|orderid|
+--------+-------+
|1       |1      |
|2       |2      |
|3       |3      |
|4       |4      |
|5       |5      |
|6       |6      |
|7       |7      |
|8       |8      |
|9       |9      |
|10      |10     |
+--------+-------+
only showing top 10 rows


In [54]:
dim_shipper = (
    shippers
    .withColumn(
        "shipper_sk",
        F.row_number().over(
            Window.orderBy("shipperid")
        )
    )
    .select(
        "shipper_sk",
        "shipperid",
        "companyname"
    )
)

In [55]:
dim_shipper.show(10, truncate=False)

+----------+---------+----------------+
|shipper_sk|shipperid|companyname     |
+----------+---------+----------------+
|1         |1        |DHL             |
|2         |2        |FedEx           |
|3         |3        |Aramex          |
|4         |4        |UPS             |
|5         |5        |Amazon Logistics|
|6         |6        |Egypt Post      |
|7         |7        |Fetchr          |
|8         |8        |Bosta           |
|9         |9        |ShipBlu         |
|10        |10       |Mylerz          |
+----------+---------+----------------+


In [56]:
dim_supplier = (
    suppliers
    .withColumn(
        "supplier_sk",
        F.row_number().over(
            Window.orderBy("supplierid")
        )
    )
    .select(
        "supplier_sk",
        "supplierid",
        "suppliername",
        "country"
    )
)

In [57]:
dim_supplier.show(10, truncate=False)

+-----------+----------+-----------------------+--------------+
|supplier_sk|supplierid|suppliername           |country       |
+-----------+----------+-----------------------+--------------+
|1          |1         |Digital World 1        |Jordan        |
|2          |2         |Global Wholesale 2     |Germany       |
|3          |3         |International Goods 3  |Jordan        |
|4          |4         |Smart Products Ltd 4   |UAE           |
|5          |5         |Prime Suppliers 5      |Germany       |
|6          |6         |Smart Products Ltd 6   |Jordan        |
|7          |7         |Future Trading 7       |UAE           |
|8          |8         |Middle East Supplies 8 |United Kingdom|
|9          |9         |Digital World 9        |Germany       |
|10         |10        |Middle East Supplies 10|United Kingdom|
+-----------+----------+-----------------------+--------------+
only showing top 10 rows


In [61]:
min_date = orders.select(F.min("orderdate")).first()[0]
max_date = orders.select(F.max("orderdate")).first()[0]

print(min_date)
print(max_date)

2024-01-01
2026-09-15


In [62]:
date_df = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(min_date),
                F.lit(max_date),
                F.expr("interval 1 day")
            )
        ).alias("full_date")
    )
)

In [63]:
date_df.show(10)

+----------+
| full_date|
+----------+
|2024-01-01|
|2024-01-02|
|2024-01-03|
|2024-01-04|
|2024-01-05|
|2024-01-06|
|2024-01-07|
|2024-01-08|
|2024-01-09|
|2024-01-10|
+----------+
only showing top 10 rows


In [66]:
dim_date = (
    date_df
    .withColumn(
        "date_sk",
        F.row_number().over(
            Window.orderBy("full_date")
        )
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .select(
        "date_sk",
        "full_date",
        "year",
        "month",
        "month_name",
        "quarter",
        "day",
        "day_of_week",
        "day_name"
    )
)

In [67]:
dim_date.show(10, truncate=False)

+-------+----------+----+-----+----------+-------+---+-----------+---------+
|date_sk|full_date |year|month|month_name|quarter|day|day_of_week|day_name |
+-------+----------+----+-----+----------+-------+---+-----------+---------+
|1      |2024-01-01|2024|1    |January   |1      |1  |2          |Monday   |
|2      |2024-01-02|2024|1    |January   |1      |2  |3          |Tuesday  |
|3      |2024-01-03|2024|1    |January   |1      |3  |4          |Wednesday|
|4      |2024-01-04|2024|1    |January   |1      |4  |5          |Thursday |
|5      |2024-01-05|2024|1    |January   |1      |5  |6          |Friday   |
|6      |2024-01-06|2024|1    |January   |1      |6  |7          |Saturday |
|7      |2024-01-07|2024|1    |January   |1      |7  |1          |Sunday   |
|8      |2024-01-08|2024|1    |January   |1      |8  |2          |Monday   |
|9      |2024-01-09|2024|1    |January   |1      |9  |3          |Tuesday  |
|10     |2024-01-10|2024|1    |January   |1      |10 |4          |Wednesday|

In [68]:
bridge_product_supplier = (
    product_suppliers
    .join(
        dim_product,
        product_suppliers.productid == dim_product.productid,
        "inner"
    )
    .join(
        dim_supplier,
        product_suppliers.supplierid == dim_supplier.supplierid,
        "inner"
    )
    .select(
        dim_product.product_sk,
        dim_supplier.supplier_sk
    )
    .dropDuplicates()
)

In [69]:
bridge_product_supplier.show(10, truncate=False)

+----------+-----------+
|product_sk|supplier_sk|
+----------+-----------+
|1         |26         |
|1         |1          |
|2         |42         |
|2         |45         |
|2         |73         |
|3         |47         |
|3         |54         |
|3         |37         |
|4         |85         |
|4         |100        |
+----------+-----------+
only showing top 10 rows


In [70]:
fact_sales = (
    order_sales
    .join(
        dim_order,
        order_sales.orderid == dim_order.orderid,
        "inner"
    )
    .join(
        dim_customer,
        order_sales.customerid == dim_customer.customerid,
        "inner"
    )
    .join(
        dim_product,
        order_sales.productid == dim_product.productid,
        "inner"
    )
    .join(
        dim_date,
        order_sales.orderdate == dim_date.full_date,
        "inner"
    )
    .select(
        dim_order.order_sk,
        dim_customer.customer_sk,
        dim_product.product_sk,
        dim_date.date_sk,
        order_sales.quantity,
        order_sales.unitprice,
        order_sales.discount,
        order_sales.total_amount
    )
)

In [71]:
fact_sales.show(10, truncate=False)

+--------+-----------+----------+-------+--------+---------+--------+------------+
|order_sk|customer_sk|product_sk|date_sk|quantity|unitprice|discount|total_amount|
+--------+-----------+----------+-------+--------+---------+--------+------------+
|48314   |1059       |532       |450    |8       |2580.41  |0       |20643.28    |
|32840   |2308       |837       |303    |3       |2227.09  |0       |6681.27     |
|2557    |762        |813       |540    |1       |358.37   |0       |358.37      |
|13069   |121        |590       |30     |3       |2971.3   |0       |8913.9      |
|43632   |7322       |999       |279    |5       |2854.19  |0       |14270.95    |
|46841   |1491       |969       |31     |7       |2243.46  |5       |15704.22    |
|23119   |4692       |92        |491    |7       |2354.66  |0       |16482.62    |
|4715    |9350       |380       |761    |7       |703.81   |0       |4926.67     |
|17690   |6584       |528       |906    |2       |1625.43  |0       |3250.86     |
|212

In [72]:
fact_payments = (
    payments
    .join(
        dim_order,
        payments.orderid == dim_order.orderid,
        "inner"
    )
    .join(
        dim_date,
        F.to_date(payments.paymentdate) == dim_date.full_date,
        "inner"
    )
    .select(
        payments.paymentid,
        dim_order.order_sk,
        dim_date.date_sk,
        payments.paymentmethod,
        payments.amount
    )
)

In [73]:
fact_payments.show(10, truncate=False)

+---------+--------+-------+-------------+-------+
|paymentid|order_sk|date_sk|paymentmethod|amount |
+---------+--------+-------+-------------+-------+
|1        |7144    |820    |Bank Transfer|1912.52|
|2        |38187   |498    |PayPal       |2520.21|
|3        |10641   |583    |Credit Card  |609.92 |
|4        |44897   |311    |Bank Transfer|4654.04|
|5        |45202   |206    |Bank Transfer|200.19 |
|6        |15114   |159    |Cash         |44.8   |
|7        |26717   |836    |Cash         |3849.76|
|8        |27343   |69     |Credit Card  |1631.14|
|9        |20619   |341    |Cash         |3859.07|
|10       |27807   |667    |Debit Card   |1484.31|
+---------+--------+-------+-------------+-------+
only showing top 10 rows


In [79]:
ship_date = dim_date.alias("ship_date")
delivery_date = dim_date.alias("delivery_date")

In [80]:
fact_shipments = (
    shipments
    .join(
        dim_order,
        shipments.orderid == dim_order.orderid,
        "inner"
    )
    .join(
        dim_shipper,
        shipments.shipperid == dim_shipper.shipperid,
        "inner"
    )
    .join(
        ship_date,
        F.to_date(shipments.shipdate) == F.col("ship_date.full_date"),
        "inner"
    )
    .join(
        delivery_date,
        F.to_date(shipments.deliverydate) == F.col("delivery_date.full_date"),
        "inner"
    )
    .select(
        shipments.shipmentid,
        dim_order.order_sk,
        dim_shipper.shipper_sk,
        F.col("ship_date.date_sk").alias("ship_date_sk"),
        F.col("delivery_date.date_sk").alias("delivery_date_sk")
    )
)

In [81]:
fact_shipments.show(10, truncate=False)

+----------+--------+----------+------------+----------------+
|shipmentid|order_sk|shipper_sk|ship_date_sk|delivery_date_sk|
+----------+--------+----------+------------+----------------+
|1         |492     |9         |256         |262             |
|2         |7899    |2         |360         |367             |
|3         |3965    |8         |353         |355             |
|4         |39416   |7         |148         |156             |
|5         |38443   |10        |928         |930             |
|6         |17469   |9         |327         |336             |
|7         |28081   |10        |433         |442             |
|8         |29203   |2         |89          |94              |
|9         |33861   |6         |262         |271             |
|10        |44858   |7         |182         |185             |
+----------+--------+----------+------------+----------------+
only showing top 10 rows


In [88]:
dim_customer.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/dim_customer")

In [89]:

dim_employee.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/dim_employee")


In [90]:
dim_product.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/dim_product")

In [91]:

dim_order.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/dim_order")

In [92]:

dim_shipper.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/dim_shipper")

In [93]:
dim_supplier.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/dim_supplier")

In [94]:
dim_date.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/dim_date")

In [95]:
bridge_product_supplier.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/bridge_product_supplier")


In [96]:
fact_sales.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/fact_sales")

In [97]:
fact_payments.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/fact_payments")


In [98]:
fact_shipments.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/fact_shipments")

In [99]:
# ==========================================
# Write Transformed Source Tables
# ==========================================

customers.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/customers")

products.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/products")

categories.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/categories")

orders.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("year", "month") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/orders")

order_details.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/order_details")

payments.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/payments")

shipments.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(f"{PROCESSED_PATH}/shipments")

print("All transformed source tables were written successfully.")

All transformed source tables were written successfully.
